# AB_Rossmann ETL Implementation Guide

## Data Warehouse / Business Intelligence Case Study

This notebook provides a step-by-step guide through the ETL (Extract, Transform, Load) process for the Rossmann Store Sales Data Warehouse project.

**Objective**: Transform raw sales data into a dimensional data model following strict transformation and validation rules.

## 1. Business Context

### Problem Statement
Rossmann (German drugstore chain) needs to optimize organizational performance through a Data Warehouse/BI solution that provides:

- **Key Performance Indicators (KPIs)**:
  - Total revenue (EUR)
  - Customer volume
  - Average spend per customer

- **Business Questions**:
  - Which store types and assortment categories generate the highest sales and customer traffic?  
  - How do promotions affect sales and customer volume? 
  - How do holidays, school holidays and day-of-week patterns influence store sales? 
  - Does the distance to competitors have an impact on store turnover and customer behavior? 

## 2. Setup and Configuration

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
import logging
import os
from sqlalchemy import create_engine, text
from datetime import datetime

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

print('Libraries imported successfully!')

Libraries imported successfully!


In [10]:
# Load environment variables from .env file
load_dotenv(override=True)

# Database connection settings (Supabase connection pooler)
SD_HOST = os.getenv('SD_HOST')
SD_PORT = os.getenv('SD_PORT')
SD_DBNAME = os.getenv('SD_DBNAME')
SD_USER = os.getenv('SD_USER')
SD_PASSWORD = os.getenv('SD_PASSWORD')
SD_SSLMODE = os.getenv('SD_SSLMODE')

# Create database connection string for PostgreSQL pooled connection
db_connection_string = f'postgresql://{SD_USER}:{SD_PASSWORD}@{SD_HOST}:{SD_PORT}/{SD_DBNAME}?sslmode={SD_SSLMODE}'
engine = create_engine(db_connection_string, echo=False, pool_size=5)

# Test database connection
try:
    with engine.connect() as connection:
        connection.execute(text('SELECT 1'))
        print('Database connection successful!')
        print(f'Connected to: {SD_HOST}:{SD_PORT}/{SD_DBNAME}')
except Exception as e:
    print(f'Database connection failed: {str(e)}')

# Set data path for CSV files
data_path = Path('data')
print(f'Data path: {data_path}')

Database connection successful!
Connected to: aws-1-eu-west-1.pooler.supabase.com:5432/postgres
Data path: data


## 3. Extract Phase - Load Raw Data

The data sources are loaded as they are in csv files.

In [11]:
def load_raw_data():
    """Load raw data from CSV files"""
    logging.info('Loading raw data from CSV files...')
    store_path = data_path / 'store.csv'
    train_path = data_path / 'train.csv'
    
    df_store = pd.read_csv(store_path)
    df_train = pd.read_csv(train_path, low_memory=False)
    
    logging.info(f'Store records loaded: {len(df_store)}')
    logging.info(f'Training records loaded: {len(df_train)}')
    
    return df_store, df_train

df_store, df_train = load_raw_data()

print('\n=== STORE DATA ===')
print(f'Shape: {df_store.shape}')
print(f'Columns: {df_store.columns.tolist()}')
print('\nFirst 3 rows:')
print(df_store.head(3))

2026-04-05 16:42:34,436 - INFO - Loading raw data from CSV files...
2026-04-05 16:42:35,334 - INFO - Store records loaded: 1115
2026-04-05 16:42:35,336 - INFO - Training records loaded: 1017209



=== STORE DATA ===
Shape: (1115, 10)
Columns: ['Store', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']

First 3 rows:
   Store StoreType Assortment  CompetitionDistance  CompetitionOpenSinceMonth  \
0      1         c          a               1270.0                        9.0   
1      2         a          a                570.0                       11.0   
2      3         a          a              14130.0                       12.0   

   CompetitionOpenSinceYear  Promo2  Promo2SinceWeek  Promo2SinceYear  \
0                    2008.0       0              NaN              NaN   
1                    2007.0       1             13.0           2010.0   
2                    2006.0       1             14.0           2011.0   

     PromoInterval  
0              NaN  
1  Jan,Apr,Jul,Oct  
2  Jan,Apr,Jul,Oct  


In [12]:
print('\n=== TRAIN DATA ===')
print(f'Shape: {df_train.shape}')
print(f'Columns: {df_train.columns.tolist()}')
print('\nFirst 3 rows:')
print(df_train.head(3))


=== TRAIN DATA ===
Shape: (1017209, 9)
Columns: ['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']

First 3 rows:
   Store  DayOfWeek        Date  Sales  Customers  Open  Promo StateHoliday  \
0      1          5  2015-07-31   5263        555     1      1            0   
1      2          5  2015-07-31   6064        625     1      1            0   
2      3          5  2015-07-31   8314        821     1      1            0   

   SchoolHoliday  
0              1  
1              1  
2              1  


## 4. Transform

### 4.1 Data Preparation

In [13]:
def validate_empty_values(df_store, df_train):
    """Check for empty/null values"""
    print('\n=== EMPTY VALUES CHECK ===')
    
    print('\nStore data missing values:')
    store_nulls = df_store.isnull().sum()
    print(store_nulls[store_nulls > 0] if (store_nulls > 0).any() else 'No missing values')
    
    print('\nTrain data missing values:')
    train_nulls = df_train.isnull().sum()
    print(train_nulls[train_nulls > 0] if (train_nulls > 0).any() else 'No missing values')
    
    # Critical validation: Date cannot be null
    if df_train['Date'].isnull().any():
        rows_with_null_dates = df_train[df_train['Date'].isnull()].shape[0]
        logging.warning(f'{rows_with_null_dates} rows have null dates and will be excluded')
        df_train = df_train.dropna(subset=['Date'])
    
    # Validation: SchoolHoliday and StateHoliday should not be null
    if df_train['SchoolHoliday'].isnull().any():
        logging.warning('Null values found in SchoolHoliday, filling with 0')
        df_train['SchoolHoliday'] = df_train['SchoolHoliday'].fillna(0)
    
    if df_train['StateHoliday'].isnull().any():
        logging.warning('Null values found in StateHoliday, filling with 0')
        df_train['StateHoliday'] = df_train['StateHoliday'].fillna('0')
    
    return df_store, df_train

df_store, df_train = validate_empty_values(df_store, df_train)


=== EMPTY VALUES CHECK ===

Store data missing values:
CompetitionDistance            3
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2SinceWeek              544
Promo2SinceYear              544
PromoInterval                544
dtype: int64

Train data missing values:
No missing values


In [14]:
print(df_store[df_store["CompetitionDistance"].isna()])

     Store StoreType Assortment  CompetitionDistance  \
290    291         d          a                  NaN   
621    622         a          c                  NaN   
878    879         d          a                  NaN   

     CompetitionOpenSinceMonth  CompetitionOpenSinceYear  Promo2  \
290                        NaN                       NaN       0   
621                        NaN                       NaN       0   
878                        NaN                       NaN       1   

     Promo2SinceWeek  Promo2SinceYear    PromoInterval  
290              NaN              NaN              NaN  
621              NaN              NaN              NaN  
878              5.0           2013.0  Feb,May,Aug,Nov  


In [15]:
# remove rows where distance is unknown
df_store = df_store[df_store["CompetitionDistance"].notna()]

### 4.2 Invalid Data Check

In [16]:
def validate_invalid_data(df_store, df_train):
    """Check for invalid data values"""
    print('\n=== INVALID DATA CHECK ===')
    
    initial_count = len(df_train)
    
    # Check for negative sales
    neg_sales = (df_train['Sales'] < 0).sum()
    if neg_sales > 0:
        logging.warning(f'{neg_sales} rows have negative sales')
        df_train = df_train[df_train['Sales'] >= 0]
    
    # Check for negative customers
    neg_customers = (df_train['Customers'] < 0).sum()
    if neg_customers > 0:
        logging.warning(f'{neg_customers} rows have negative customers')
        df_train = df_train[df_train['Customers'] >= 0]
    
    # Check for negative competition distance
    neg_distance = (df_store['CompetitionDistance'] < 0).sum()
    if neg_distance > 0:
        logging.warning(f'{neg_distance} stores have negative competition distance')
        df_store = df_store[df_store['CompetitionDistance'] >= 0]
    
    # Check for future dates
    df_train['Date'] = pd.to_datetime(df_train['Date'])
    future_dates = (df_train['Date'] > datetime.now()).sum()
    if future_dates > 0:
        logging.warning(f'{future_dates} rows have future dates')
        df_train = df_train[df_train['Date'] <= datetime.now()]
    
    # Check promotion year is not greater than date year
    df_store['Promo2SinceYear'] = pd.to_numeric(df_store['Promo2SinceYear'], errors='coerce')
    invalid_promo_years = 0
    # This will be validated after join
    
    rows_removed = initial_count - len(df_train)
    if rows_removed > 0:
        logging.info(f'Removed {rows_removed} rows due to invalid data')
    else:
        logging.info(f'No invalid data found therefore no rows were removed.')
    
    return df_store, df_train

df_store, df_train = validate_invalid_data(df_store, df_train)

2026-04-05 16:42:48,851 - INFO - No invalid data found therefore no rows were removed.



=== INVALID DATA CHECK ===


### 4.3 Missing Keys Check

In [17]:
def validate_missing_keys(df_store, df_train):
    """Check if Store ID is present in both sources"""
    print('\n=== MISSING KEYS CHECK ===')
    
    store_ids_in_store = set(df_store['Store'].unique())
    store_ids_in_train = set(df_train['Store'].unique())
    
    missing_in_store = store_ids_in_train - store_ids_in_store
    missing_in_train = store_ids_in_store - store_ids_in_train
    
    if missing_in_store:
        logging.warning(f'Store IDs in train but not in store: {len(missing_in_store)} stores')
        df_train = df_train[~df_train['Store'].isin(missing_in_store)]
    
    if missing_in_train:
        logging.warning(f'Store IDs in store but not in train: {len(missing_in_train)} stores')
    
    print(f'Total stores in store dataset: {len(store_ids_in_store)}')
    print(f'Stores with training data: {len(store_ids_in_store - missing_in_train)}')
    
    return df_store, df_train

df_store, df_train = validate_missing_keys(df_store, df_train)

2026-04-05 16:42:52,089 - WARNING - Store IDs in train but not in store: 3 stores



=== MISSING KEYS CHECK ===
Total stores in store dataset: 1112
Stores with training data: 1112


### 4.4 Duplicate Check

In [18]:
def validate_duplicates(df_train):
    """Check for duplicate entries"""
    print('\n=== DUPLICATE CHECK ===')
    
    # Check for duplicate Date + Store combinations
    duplicates_date_store = df_train.duplicated(subset=['Date', 'Store']).sum()
    if duplicates_date_store > 0:
        logging.warning(f'{duplicates_date_store} duplicate Date + Store combinations found')
        df_train = df_train.drop_duplicates(subset=['Date', 'Store'], keep='first')
    else:
        print('No duplicate Date + Store combinations found')
    
    print(f'Training records after duplicate removal: {len(df_train)}')
    
    return df_train

df_train = validate_duplicates(df_train)


=== DUPLICATE CHECK ===
No duplicate Date + Store combinations found
Training records after duplicate removal: 1014567


## 5. Data Preparation Phase

### 5.1 Join Data Sources

In [19]:
def prepare_data(df_store, df_train):
    """Join store and train datasets"""
    print('\n=== DATA PREPARATION - JOIN ===')
    
    # Join datasets on Store ID
    df_merged = df_train.merge(df_store, left_on='Store', right_on='Store', how='inner')
    
    print(f'Records after join: {len(df_merged)}')
    print(f'Columns in merged data: {len(df_merged.columns)}')
    
    return df_merged

df_merged = prepare_data(df_store, df_train)
print('\nMerged data sample:')
print(df_merged.head(2))


=== DATA PREPARATION - JOIN ===
Records after join: 1014567
Columns in merged data: 18

Merged data sample:
   Store  DayOfWeek       Date  Sales  Customers  Open  Promo StateHoliday  \
0      1          5 2015-07-31   5263        555     1      1            0   
1      2          5 2015-07-31   6064        625     1      1            0   

   SchoolHoliday StoreType Assortment  CompetitionDistance  \
0              1         c          a               1270.0   
1              1         a          a                570.0   

   CompetitionOpenSinceMonth  CompetitionOpenSinceYear  Promo2  \
0                        9.0                    2008.0       0   
1                       11.0                    2007.0       1   

   Promo2SinceWeek  Promo2SinceYear    PromoInterval  
0              NaN              NaN              NaN  
1             13.0           2010.0  Jan,Apr,Jul,Oct  


### 5.2 Standardize Data Types and Rename Columns

In [20]:
def standardize_and_rename(df):
    """Standardize data types and rename columns"""
    print('\n=== STANDARDIZE DATA TYPES AND RENAME ===')
    
    df = df.copy()
    
    # Standardize Date format to YYYY-MM-DD
    df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')
    
    # Convert CompetitionDistance to float
    df['CompetitionDistance'] = pd.to_numeric(df['CompetitionDistance'], errors='coerce').astype('float64')
    
    # Convert Customers to integer
    df['Customers'] = pd.to_numeric(df['Customers'], errors='coerce').astype('int64')
    
    # Convert Sales to float
    df['Sales'] = pd.to_numeric(df['Sales'], errors='coerce').astype('float64')
    
    # Rename columns
    rename_dict = {
        'Store': 'store_id',
        'StoreType': 'store_type',
        'Assortment': 'assortment',
        'Promo2': 'promotion',
        'Promo2SinceYear': 'promotion_year',
        'PromoInterval': 'promotion_interval',
        'CompetitionDistance': 'distance',
        'CompetitionOpenSinceYear': 'comp_open_year',
        'CompetitionOpenSinceMonth': 'comp_open_month',
        'Date': 'full_date',
        'StateHoliday': 'state_holiday',
        'SchoolHoliday': 'school_holiday',
        'Sales': 'turnover',
        'Customers': 'nr_customers'
    }
    
    df = df.rename(columns=rename_dict)
    
    print('Column renaming completed')
    print(f'Columns after rename: {[c for c in df.columns if c in rename_dict.values()]}')
    
    return df

df_merged = standardize_and_rename(df_merged)


=== STANDARDIZE DATA TYPES AND RENAME ===
Column renaming completed
Columns after rename: ['store_id', 'full_date', 'turnover', 'nr_customers', 'state_holiday', 'school_holiday', 'store_type', 'assortment', 'distance', 'comp_open_month', 'comp_open_year', 'promotion', 'promotion_year', 'promotion_interval']


### 5.3 Map Holiday Values

In [21]:
def map_holiday_values(df):
    """Map state holiday values to integers"""
    print('\n=== MAP HOLIDAY VALUES ===')
    
    df = df.copy()
    
    # Map StateHoliday: 0=None, a=Public(1), b=Easter(2), c=Christmas(3)
    state_holiday_map = {'0': 0, 'a': 1, 'b': 2, 'c': 3}
    df['state_holiday'] = df['state_holiday'].astype(str).map(state_holiday_map).fillna(0).astype('int64')
    
    # Ensure SchoolHoliday is binary (0 or 1)
    df['school_holiday'] = df['school_holiday'].astype('int64')
    
    print('Holiday values mapped')
    print(f'State holiday unique values: {sorted(df["state_holiday"].unique())}')
    print(f'School holiday unique values: {sorted(df["school_holiday"].unique())}')
    
    return df

df_merged = map_holiday_values(df_merged)


=== MAP HOLIDAY VALUES ===
Holiday values mapped
State holiday unique values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
School holiday unique values: [np.int64(0), np.int64(1)]


## 6. Transform Phase - Create Dimension Tables

### 6.1 Create dim_store

In [22]:
def create_dim_store(df):
    """Create store dimension"""
    print('\n=== CREATE DIM_STORE ===')
    
    dim_store = df[['store_id', 'store_type', 'assortment']].drop_duplicates().reset_index(drop=True)
    
    # Map assortment codes to full names
    assort_map = {'a': 'Basic', 'b': 'Extra', 'c': 'Extended'}
    dim_store['assortment'] = dim_store['assortment'].map(assort_map)
    
    # Validate assortment
    if dim_store['assortment'].isnull().any():
        logging.warning(f'{dim_store["assortment"].isnull().sum()} rows have invalid assortment')
        dim_store = dim_store.dropna(subset=['assortment'])
    
    dim_store = dim_store.sort_values('store_id').reset_index(drop=True)
    
    print(f'dim_store records: {len(dim_store)}')
    print(f'Store types: {dim_store["store_type"].unique()}')
    print(f'Assortments: {dim_store["assortment"].unique()}')
    
    return dim_store

dim_store = create_dim_store(df_merged)
print('\nFirst 5 records:')
print(dim_store.head())


=== CREATE DIM_STORE ===
dim_store records: 1112
Store types: <StringArray>
['c', 'a', 'd', 'b']
Length: 4, dtype: str
Assortments: <StringArray>
['Basic', 'Extended', 'Extra']
Length: 3, dtype: str

First 5 records:
   store_id store_type assortment
0         1          c      Basic
1         2          a      Basic
2         3          a      Basic
3         4          c   Extended
4         5          a      Basic


### 6.2 Create dim_competition

In [23]:
def create_dim_competition(df):
    """Create competition dimension"""
    print('\n=== CREATE DIM_COMPETITION ===')
    
    comp_data = []
    
    for store_id in df['store_id'].unique():
        store_data = df[df['store_id'] == store_id].iloc[0]
        
        # Determine if competitor is open
        is_open = 1 if pd.notna(store_data['comp_open_year']) else 0
        
        comp_data.append({
            'distance': float(store_data['distance']) if pd.notna(store_data['distance']) else 0.0,
            'open': is_open
        })
    
    dim_competition = pd.DataFrame(comp_data).reset_index(drop=True)
    
    print(f'dim_competition records: {len(dim_competition)}')
    print(f'\nCompetition distance statistics:')
    print(dim_competition['distance'].describe())
    
    return dim_competition

dim_competition = create_dim_competition(df_merged)
print('\nFirst 5 records:')
print(dim_competition.head())


=== CREATE DIM_COMPETITION ===
dim_competition records: 1112

Competition distance statistics:
count     1112.000000
mean      5404.901079
std       7663.174720
min         20.000000
25%        717.500000
50%       2325.000000
75%       6882.500000
max      75860.000000
Name: distance, dtype: float64

First 5 records:
   distance  open
0    1270.0     1
1     570.0     1
2   14130.0     1
3     620.0     1
4   29910.0     1


### 6.3 Create dim_promotion with PromoInterval splitting

In [24]:
def create_dim_promotion(df):
    """Create promotion dimension with one row per PromoInterval month"""
    print('\n=== CREATE DIM_PROMOTION (one row per month) ===')
    
    promo_data = []
    
    for store_id in df['store_id'].unique():
        store_data = df[df['store_id'] == store_id].iloc[0]
        
        has_promotion = int(store_data['promotion']) if pd.notna(store_data['promotion']) else 0
        promo_year = int(store_data['promotion_year']) if pd.notna(store_data['promotion_year']) else 0
        promo_interval = str(store_data['promotion_interval']) if pd.notna(store_data['promotion_interval']) else ''
        
        if has_promotion == 0 or promo_interval == '' or promo_interval == 'nan':
            promo_data.append({
                'promotion': has_promotion,
                'promotion_year': promo_year,
                'promotion_interval': None
            })
        else:
            months = [m.strip() for m in promo_interval.split(',')]
            
            for month in months:
                promo_data.append({
                    'promotion': has_promotion,
                    'promotion_year': promo_year,
                    'promotion_interval': month
                })
    
    dim_promotion = pd.DataFrame(promo_data).reset_index(drop=True)
    
    print(f'dim_promotion records: {len(dim_promotion)}')
    print(f'Rows with active promotions: {(dim_promotion["promotion"] == 1).sum()}')
    print('\nUnique promotion months:')
    print(dim_promotion['promotion_interval'].dropna().unique())
    
    return dim_promotion

dim_promotion = create_dim_promotion(df_merged)
print('\nFirst 10 records:')
print(dim_promotion.head(10))


=== CREATE DIM_PROMOTION (one row per month) ===
dim_promotion records: 2822
Rows with active promotions: 2280

Unique promotion months:
<StringArray>
[ 'Jan',  'Apr',  'Jul',  'Oct',  'Feb',  'May',  'Aug',  'Nov',  'Mar',
  'Jun', 'Sept',  'Dec']
Length: 12, dtype: str

First 10 records:
   promotion  promotion_year promotion_interval
0          0               0                NaN
1          1            2010                Jan
2          1            2010                Apr
3          1            2010                Jul
4          1            2010                Oct
5          1            2011                Jan
6          1            2011                Apr
7          1            2011                Jul
8          1            2011                Oct
9          0               0                NaN


### 6.4 Create dim_time with date derivations

In [43]:
def create_dim_time(df):
    """Create time dimension with date derivations and time_id"""
    print('\n=== CREATE DIM_TIME ===')
    
    # Get unique dates
    unique_dates = pd.to_datetime(df['full_date']).dropna().unique()
    unique_dates = sorted(unique_dates)
    
    time_data = []
    
    for i, date in enumerate(unique_dates, start=1):
        day_data = df[pd.to_datetime(df['full_date']) == date]
        
        # Get holiday info
        school_holiday = int(day_data['school_holiday'].iloc[0])
        state_holiday = int(day_data['state_holiday'].iloc[0])
        
        # Validate holiday values
        school_holiday = 1 if school_holiday > 0 else 0
        state_holiday = state_holiday if state_holiday in [0, 1, 2, 3] else 0
        
        time_data.append({
            'time_id': i,
            'full_date': pd.Timestamp(date).strftime('%Y-%m-%d'),
            'year': pd.Timestamp(date).year,
            'month': pd.Timestamp(date).month,
            'day': pd.Timestamp(date).day,
            'school_holiday': school_holiday,
            'state_holiday': state_holiday
        })
    
    dim_time = pd.DataFrame(time_data).reset_index(drop=True)
    
    print(f'dim_time records: {len(dim_time)}')
    print(f'Date range: {dim_time["full_date"].min()} to {dim_time["full_date"].max()}')
    print(f'School holidays: {(dim_time["school_holiday"] == 1).sum()}')
    print(f'State holidays: {(dim_time["state_holiday"] > 0).sum()}')
    
    return dim_time


dim_time = create_dim_time(df_merged)
print('\nFirst 10 records:')
print(dim_time.head(10))


=== CREATE DIM_TIME ===
dim_time records: 942
Date range: 2013-01-01 to 2015-07-31
School holidays: 193
State holidays: 27

First 10 records:
   time_id   full_date  year  month  day  school_holiday  state_holiday
0        1  2013-01-01  2013      1    1               1              1
1        2  2013-01-02  2013      1    2               1              0
2        3  2013-01-03  2013      1    3               1              0
3        4  2013-01-04  2013      1    4               1              0
4        5  2013-01-05  2013      1    5               1              0
5        6  2013-01-06  2013      1    6               1              0
6        7  2013-01-07  2013      1    7               1              0
7        8  2013-01-08  2013      1    8               1              0
8        9  2013-01-09  2013      1    9               1              0
9       10  2013-01-10  2013      1   10               1              0


## 7. Create Fact Table with Business Rules

In [59]:
def create_fact_sales(df, dim_store, dim_competition, dim_promotion, dim_time):
    """Create fact table with business rules applied"""
    print('\n=== CREATE FACT_SALES TABLE ===')
    
    df = df.copy()
    
    # Make sure date columns are datetime
    df['full_date'] = pd.to_datetime(df['full_date'])
    dim_time = dim_time.copy()
    dim_time['full_date'] = pd.to_datetime(dim_time['full_date'])
    
    # Create mappings from dimension tables
    store_id_map = {row['store_id']: row['store_id'] for _, row in dim_store.iterrows()}
    
    # Competition and promotion mapping
    # Assumption: dim_competition and dim_promotion were created in store_id order
    unique_store_ids = sorted(df['store_id'].dropna().unique())
    competition_id_map = {store_id: idx + 1 for idx, store_id in enumerate(unique_store_ids)}
    promotion_id_map = {store_id: idx + 1 for idx, store_id in enumerate(unique_store_ids)}
    
    # Time mapping from dim_time
    time_id_map = dict(zip(dim_time['full_date'], dim_time['time_id']))
    
    # Create fact table
    fact = pd.DataFrame()
    fact['store_id'] = df['store_id']
    
    # Add dimension IDs
    fact['competition_id'] = df['store_id'].map(competition_id_map)
    fact['promotion_id'] = df['store_id'].map(promotion_id_map)
    fact['time_id'] = df['full_date'].map(time_id_map)
    
    # Add measures
    fact['turnover'] = df['turnover']
    fact['nr_customers'] = df['nr_customers']
    
    # Apply business rule: Calculate turnover_per_customer
    fact['turnover_per_customer'] = 0.0
    mask = fact['nr_customers'] > 0
    fact.loc[mask, 'turnover_per_customer'] = (
        (fact.loc[mask, 'turnover'] / fact.loc[mask, 'nr_customers']).round(2)
    )
    
    # Apply business rules
    print('\nApplying business rules...')
    
    # Rule 1: Turnover must be >= 0
    neg_turnover = (fact['turnover'] < 0).sum()
    if neg_turnover > 0:
        logging.warning(f'{neg_turnover} rows have negative turnover')
        fact = fact[fact['turnover'] >= 0]
    
    # Rule 2: nr_customers must be >= 0
    neg_customers = (fact['nr_customers'] < 0).sum()
    if neg_customers > 0:
        logging.warning(f'{neg_customers} rows have negative customers')
        fact = fact[fact['nr_customers'] >= 0]
    
    # Rule 3: If nr_customers = 0, turnover_per_customer should be 0
    fact.loc[fact['nr_customers'] == 0, 'turnover_per_customer'] = 0.0
    
    # Rule 4: Each fact row must reference valid dimension keys
    # Validate store_id exists in dim_store
    valid_store_ids = set(dim_store['store_id'])
    invalid_stores = (~fact['store_id'].isin(valid_store_ids)).sum()
    if invalid_stores > 0:
        logging.warning(f'{invalid_stores} rows have invalid store_id')
        fact = fact[fact['store_id'].isin(valid_store_ids)]
    
    # Validate competition_id
    valid_competition_ids = set(range(1, len(dim_competition) + 1))
    invalid_competition = (~fact['competition_id'].isin(valid_competition_ids)).sum()
    if invalid_competition > 0:
        logging.warning(f'{invalid_competition} rows have invalid competition_id')
        fact = fact[fact['competition_id'].isin(valid_competition_ids)]
    
    # Validate promotion_id
    valid_promotion_ids = set(range(1, len(dim_promotion) + 1))
    invalid_promotion = (~fact['promotion_id'].isin(valid_promotion_ids)).sum()
    if invalid_promotion > 0:
        logging.warning(f'{invalid_promotion} rows have invalid promotion_id')
        fact = fact[fact['promotion_id'].isin(valid_promotion_ids)]
    
    # Validate time_id
    valid_time_ids = set(dim_time['time_id'])
    invalid_time = (~fact['time_id'].isin(valid_time_ids)).sum()
    if invalid_time > 0:
        logging.warning(f'{invalid_time} rows have invalid time_id')
        fact = fact[fact['time_id'].isin(valid_time_ids)]
    
    # Convert IDs to int after filtering
    fact['store_id'] = fact['store_id'].astype(int)
    fact['competition_id'] = fact['competition_id'].astype(int)
    fact['promotion_id'] = fact['promotion_id'].astype(int)
    fact['time_id'] = fact['time_id'].astype(int)
    
    print(f'\nfact_sales records: {len(fact)}')
    print(f'Total turnover: EUR {fact["turnover"].sum():,.2f}')
    print(f'Total customers: {fact["nr_customers"].sum():,}')
    print(f'Average turnover per transaction: EUR {fact["turnover"].mean():,.2f}')
    print(f'Average turnover per customer: EUR {fact[fact["nr_customers"] > 0]["turnover_per_customer"].mean():,.2f}')
    
    return fact

fact_sales = create_fact_sales(df_merged, dim_store, dim_competition, dim_promotion, dim_time)
print('\nFirst 10 records:')
print(fact_sales.head(10))



=== CREATE FACT_SALES TABLE ===

Applying business rules...

fact_sales records: 1014567
Total turnover: EUR 5,861,196,794.00
Total customers: 642,833,125
Average turnover per transaction: EUR 5,777.04
Average turnover per customer: EUR 9.49

First 10 records:
   store_id  competition_id  promotion_id  time_id  turnover  nr_customers  \
0         1               1             1      942    5263.0           555   
1         2               2             2      942    6064.0           625   
2         3               3             3      942    8314.0           821   
3         4               4             4      942   13995.0          1498   
4         5               5             5      942    4822.0           559   
5         6               6             6      942    5651.0           589   
6         7               7             7      942   15344.0          1414   
7         8               8             8      942    8492.0           833   
8         9               9         

## 8. Data Quality Summary

In [27]:
print('\n' + '='*70)
print('DATA QUALITY AND TRANSFORMATION SUMMARY')
print('='*70)

print(f'\nDimension Tables:')
print(f'  dim_store:        {len(dim_store):6} records')
print(f'  dim_competition:  {len(dim_competition):6} records')
print(f'  dim_promotion:    {len(dim_promotion):6} records')
print(f'  dim_time:         {len(dim_time):6} records')

print(f'\nFact Table:')
print(f'  fact_sales:       {len(fact_sales):6} records')

print(f'\nData Quality Checks:')
print(f'  No null store_id:           {(fact_sales["store_id"].notnull().all())}')
print(f'  No negative turnover:       {(fact_sales["turnover"] >= 0).all()}')
print(f'  No negative customers:      {(fact_sales["nr_customers"] >= 0).all()}')
print(f'  Valid turnover_per_customer:{(fact_sales["turnover_per_customer"] >= 0).all()}')

print(f'\nDimension Table Checks:')
print(f'  No null values in dim_store:      {dim_store.isnull().sum().sum() == 0}')
print(f'  No null values in dim_time:       {dim_time.isnull().sum().sum() == 0}')
print(f'  No null values in dim_competition:{dim_competition.isnull().sum().sum() == 0}')
print(f'  No null values in dim_promotion:  {dim_promotion.isnull().sum().sum() == 0}')

print('\n' + '='*70)


DATA QUALITY AND TRANSFORMATION SUMMARY

Dimension Tables:
  dim_store:          1112 records
  dim_competition:    1112 records
  dim_promotion:      2822 records
  dim_time:            942 records

Fact Table:
  fact_sales:       1014567 records

Data Quality Checks:
  No null store_id:           True
  No negative turnover:       True
  No negative customers:      True
  Valid turnover_per_customer:True

Dimension Table Checks:
  No null values in dim_store:      True
  No null values in dim_time:       True
  No null values in dim_competition:True
  No null values in dim_promotion:  False



## 9. Load Phase - Upload to Database

In [28]:
def upload_to_database(df, table_name, if_exists='append', chunksize=1000):
    """Upload dataframe to PostgreSQL database in batches"""
    try:
        logging.info(f'Uploading {len(df)} rows to {table_name}...')
        df.to_sql(
            table_name,
            engine,
            if_exists=if_exists,
            index=False,
            method='multi',
            chunksize=chunksize
        )
        logging.info(f'Successfully uploaded {len(df)} rows to {table_name}')
        return True
    except Exception as e:
        logging.error(f'Error uploading to {table_name}: {str(e)}')
        raise

print('Upload function defined.')

Upload function defined.


In [29]:
def clear_table(table_name):
    """Clear all records from a table"""
    try:
        with engine.connect() as connection:
            connection.execute(text(f'DELETE FROM {table_name}'))
            connection.commit()
            logging.info(f'Cleared table: {table_name}')
    except Exception as e:
        logging.warning(f'Could not clear table {table_name}: {str(e)}')

print('Clear function defined.')

Clear function defined.


In [61]:
# Upload dimensions
print('\nUploading dimension tables...')
upload_to_database(dim_store, 'dim_store', if_exists='append')
upload_to_database(dim_competition, 'dim_competition', if_exists='append')
upload_to_database(dim_promotion, 'dim_promotion', if_exists='append')
upload_to_database(dim_time, 'dim_time', if_exists='append')
print('Dimension tables uploaded')

# Upload fact table
print('\nUploading fact table...')
upload_to_database(fact_sales, 'fact_sales', if_exists='append', chunksize=1000)
print('Fact table uploaded')



2026-04-05 17:22:54,846 - INFO - Uploading 1112 rows to dim_store...



Uploading dimension tables...


2026-04-05 17:22:55,251 - INFO - Successfully uploaded 1112 rows to dim_store
2026-04-05 17:22:55,252 - INFO - Uploading 1112 rows to dim_competition...
2026-04-05 17:22:55,417 - INFO - Successfully uploaded 1112 rows to dim_competition
2026-04-05 17:22:55,419 - INFO - Uploading 2822 rows to dim_promotion...
2026-04-05 17:22:55,758 - INFO - Successfully uploaded 2822 rows to dim_promotion
2026-04-05 17:22:55,759 - INFO - Uploading 942 rows to dim_time...
2026-04-05 17:22:55,961 - INFO - Successfully uploaded 942 rows to dim_time
2026-04-05 17:22:55,962 - INFO - Uploading 1014567 rows to fact_sales...


Dimension tables uploaded

Uploading fact table...


2026-04-05 17:26:21,937 - INFO - Successfully uploaded 1014567 rows to fact_sales


Fact table uploaded


## 10. Complete ETL Workflow

In [ ]:
def run_complete_etl():
    """Run complete ETL process with all validations and transformations"""
    try:
        print('\n' + '='*70)
        print('COMPLETE ETL PIPELINE - EXTRACT, VALIDATE, TRANSFORM, LOAD')
        print('='*70)
        
        # EXTRACT
        print('\n>>> EXTRACT PHASE')
        df_store, df_train = load_raw_data()
        
        # VALIDATE
        print('\n>>> VALIDATION PHASE')
        df_store, df_train = validate_empty_values(df_store, df_train)
        df_store, df_train = validate_invalid_data(df_store, df_train)
        df_store, df_train = validate_missing_keys(df_store, df_train)
        df_train = validate_duplicates(df_train)
        
        # PREPARE
        print('\n>>> DATA PREPARATION PHASE')
        df_merged = prepare_data(df_store, df_train)
        df_merged = standardize_and_rename(df_merged)
        df_merged = map_holiday_values(df_merged)
        
        # TRANSFORM
        print('\n>>> TRANSFORM PHASE')
        dim_store = create_dim_store(df_merged)
        dim_competition = create_dim_competition(df_merged)
        dim_promotion = create_dim_promotion(df_merged)
        dim_time = create_dim_time(df_merged)
        fact_sales = create_fact_sales(df_merged, dim_store, dim_competition, dim_promotion, dim_time)
        
        # LOAD
        print('\n>>> LOAD PHASE')
        upload_to_database(dim_store, 'dim_store', if_exists='replace')
        upload_to_database(dim_competition, 'dim_competition', if_exists='replace')
        upload_to_database(dim_promotion, 'dim_promotion', if_exists='replace')
        upload_to_database(dim_time, 'dim_time', if_exists='replace')
        upload_to_database(fact_sales, 'fact_sales', if_exists='replace')
        
        print('\n' + '='*70)
        print('ETL PIPELINE COMPLETED SUCCESSFULLY')
        print('='*70)
        
    except Exception as e:
        logging.error(f'ETL process failed: {str(e)}')
        raise e

# Uncomment to run the complete ETL pipeline
# run_complete_etl()

print('Complete ETL pipeline function defined.')
print('Uncomment run_complete_etl() to execute the full pipeline.')

## 11. Verification and Analysis

In [ ]:
print('\n' + '='*70)
print('BUSINESS ANALYSIS AND METRICS')
print('='*70)

print(f'\nSales Summary:')
print(f'  Total turnover:             EUR {fact_sales["turnover"].sum():,.2f}')
print(f'  Total customers:            {fact_sales["nr_customers"].sum():,}')
print(f'  Average turnover per day:   EUR {fact_sales["turnover"].mean():,.2f}')
print(f'  Average customers per day:  {fact_sales["nr_customers"].mean():.0f}')
print(f'  Average per customer:       EUR {fact_sales[fact_sales["nr_customers"] > 0]["turnover_per_customer"].mean():,.2f}')

print(f'\nStore Analysis:')
print(f'  Total stores:               {len(dim_store)}')
print(f'  Store types:                {dim_store["store_type"].nunique()}')
print(f'  Assortments:                {dim_store["assortment"].nunique()}')

print(f'\nCompetition Analysis:')
print(f'  Avg distance to competitor: {dim_competition["distance"].mean():.0f}m')
print(f'  Stores with competitors:    {(dim_competition["open"] == 1).sum()}')

print(f'\nPromotion Analysis:')
print(f'  Stores with promotions:     {(dim_promotion["promotion"] == 1).sum()}')

print(f'\nTime Period Analysis:')
print(f'  Date range:                 {dim_time["full_date"].min()} to {dim_time["full_date"].max()}')
print(f'  Total days:                 {len(dim_time)}')
print(f'  School holidays:            {(dim_time["school_holiday"] == 1).sum()} days')
print(f'  State holidays:             {(dim_time["state_holiday"] > 0).sum()} days')